# Active Learning for Fake News Detection with Domain Adaptation

## Project Overview
This notebook implements Active Learning strategies for fake news detection with domain adaptation:
- **Domain A**: General news articles (Fake.csv + True.csv)
- **Domain B**: COVID-19 related news articles (Covid.csv)

## Objectives
1. Train a baseline model on Domain A
2. Evaluate the impact of domain shift on Domain B
3. Implement and compare Active Learning strategies for domain adaptation

## 1. Setup and Imports

In [5]:
# Install required packages
!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm

# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # AdamW est maintenant dans torch.optim
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import random
import time
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Data Loading and Preprocessing

In [6]:
def load_domain_a_data(fake_path, true_path):
    """Load and combine Fake and True news data for Domain A"""
    # Load data with proper handling
    fake_df = pd.read_csv(fake_path, encoding='utf-8', on_bad_lines='skip')
    true_df = pd.read_csv(true_path, encoding='utf-8', on_bad_lines='skip')
    
    # Clean column names (remove BOM if present)
    fake_df.columns = [col.replace('\ufeff', '') for col in fake_df.columns]
    true_df.columns = [col.replace('\ufeff', '') for col in true_df.columns]
    
    # Add labels
    fake_df['label'] = 1  # Fake = 1
    true_df['label'] = 0  # Real = 0
    
    # Combine title and text
    fake_df['combined_text'] = fake_df['title'].fillna('') + ' ' + fake_df['text'].fillna('')
    true_df['combined_text'] = true_df['title'].fillna('') + ' ' + true_df['text'].fillna('')
    
    # Combine both datasets
    domain_a = pd.concat([fake_df[['combined_text', 'label']], 
                          true_df[['combined_text', 'label']]], ignore_index=True)
    
    return domain_a

def load_domain_b_data(covid_path):
    """Load COVID-19 news data for Domain B"""
    covid_df = pd.read_csv(covid_path, encoding='utf-8')
    
    # Combine title and text
    covid_df['combined_text'] = covid_df['title'].fillna('') + ' ' + covid_df['text'].fillna('')
    
    # Select relevant columns (label: 0 = Real, 1 = Fake)
    domain_b = covid_df[['combined_text', 'label']].copy()
    
    return domain_b

# Load data
domain_a = load_domain_a_data('Fake.csv', 
                              'True.csv')
domain_b = load_domain_b_data('Covid.csv')

print(f"Domain A (General News): {len(domain_a)} samples")
print(f"  - Fake: {sum(domain_a['label'] == 1)}, Real: {sum(domain_a['label'] == 0)}")
print(f"\nDomain B (COVID-19): {len(domain_b)} samples")
print(f"  - Fake: {sum(domain_b['label'] == 1)}, Real: {sum(domain_b['label'] == 0)}")

Domain A (General News): 2328 samples
  - Fake: 1152, Real: 1176

Domain B (COVID-19): 3119 samples
  - Fake: 2061, Real: 1058


## 3. Dataset Splitting

In [7]:
def split_domain_data(domain_data, train_ratio=0.4, test_ratio=0.6, random_state=42):
    """Split domain data into train and test sets with stratification"""
    train_df, test_df = train_test_split(
        domain_data, 
        train_size=train_ratio,
        test_size=test_ratio,
        stratify=domain_data['label'],
        random_state=random_state
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

# Split Domain A into Train_A and Test_A
train_a, test_a = split_domain_data(domain_a, train_ratio=0.5, test_ratio=0.5)

# Split Domain B into Pool AL and Test_B
pool_b, test_b = split_domain_data(domain_b, train_ratio=0.65, test_ratio=0.35)

# Display split information
print("=" * 60)
print("DATASET SPLIT SUMMARY")
print("=" * 60)
print(f"\nDomain A (General News):")
print(f"  Train_A: {len(train_a)} samples")
print(f"    - Fake: {sum(train_a['label'] == 1)}, Real: {sum(train_a['label'] == 0)}")
print(f"  Test_A: {len(test_a)} samples")
print(f"    - Fake: {sum(test_a['label'] == 1)}, Real: {sum(test_a['label'] == 0)}")

print(f"\nDomain B (COVID-19):")
print(f"  Pool AL: {len(pool_b)} samples")
print(f"    - Fake: {sum(pool_b['label'] == 1)}, Real: {sum(pool_b['label'] == 0)}")
print(f"  Test_B: {len(test_b)} samples")
print(f"    - Fake: {sum(test_b['label'] == 1)}, Real: {sum(test_b['label'] == 0)}")

DATASET SPLIT SUMMARY

Domain A (General News):
  Train_A: 1164 samples
    - Fake: 576, Real: 588
  Test_A: 1164 samples
    - Fake: 576, Real: 588

Domain B (COVID-19):
  Pool AL: 2027 samples
    - Fake: 1339, Real: 688
  Test_B: 1092 samples
    - Fake: 722, Real: 370


In [8]:
# Create summary table for report
split_summary = pd.DataFrame({
    'Domain': ['Domain A', 'Domain A', 'Domain B', 'Domain B'],
    'Partition': ['Train_A', 'Test_A', 'Pool AL', 'Test_B'],
    'Fake News': [sum(train_a['label'] == 1), sum(test_a['label'] == 1), 
                  sum(pool_b['label'] == 1), sum(test_b['label'] == 1)],
    'Real News': [sum(train_a['label'] == 0), sum(test_a['label'] == 0),
                  sum(pool_b['label'] == 0), sum(test_b['label'] == 0)],
    'Total': [len(train_a), len(test_a), len(pool_b), len(test_b)]
})

print("\nTable 1: Dataset Split Distribution")
print(split_summary.to_string(index=False))


Table 1: Dataset Split Distribution
  Domain Partition  Fake News  Real News  Total
Domain A   Train_A        576        588   1164
Domain A    Test_A        576        588   1164
Domain B   Pool AL       1339        688   2027
Domain B    Test_B        722        370   1092


## 4. Model Definition

In [9]:
class FakeNewsDataset(Dataset):
    """Custom Dataset for Fake News Classification"""
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

def count_parameters(model):
    """Count trainable parameters in model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Training and Evaluation Functions

In [10]:
def train_model(model, train_loader, optimizer, device, epochs=3):
    """Train the model"""
    model.train()
    model.to(device)
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}')
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})
    
    return model

def evaluate_model(model, test_loader, device):
    """Evaluate the model and return metrics"""
    model.eval()
    model.to(device)
    
    all_predictions = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            predictions = torch.argmax(outputs.logits, dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    metrics = {
        'accuracy': accuracy_score(all_labels, all_predictions),
        'f1': f1_score(all_labels, all_predictions),
        'precision': precision_score(all_labels, all_predictions),
        'recall': recall_score(all_labels, all_predictions)
    }
    
    return metrics, np.array(all_predictions), np.array(all_labels), np.array(all_probs)

## 6. Baseline Model Training on Domain A

In [ ]:
# Initialize tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

print(f"Model: DistilBERT")
print(f"Number of trainable parameters: {count_parameters(model):,}")

# Create datasets and dataloaders
train_dataset_a = FakeNewsDataset(train_a['combined_text'].values, train_a['label'].values, tokenizer)
test_dataset_a = FakeNewsDataset(test_a['combined_text'].values, test_a['label'].values, tokenizer)

train_loader_a = DataLoader(train_dataset_a, batch_size=16, shuffle=True)
test_loader_a = DataLoader(test_dataset_a, batch_size=16, shuffle=False)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Train baseline model on Domain A
print("\n" + "="*60)
print("TRAINING BASELINE MODEL ON DOMAIN A")
print("="*60)

start_time = time.time()
model = train_model(model, train_loader_a, optimizer, device, epochs=3)
training_time = time.time() - start_time

print(f"\nTraining time: {training_time:.2f} seconds")

Loading weights: 100%|██| 100/100 [00:00<00:00, 173.04it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: DistilBERT
Number of trainable parameters: 66,955,010

TRAINING BASELINE MODEL ON DOMAIN A


Epoch 1/3:   0%|                                                                                             | 0/73 [00:00<?, ?it/s]

## 7. Baseline Evaluation on Both Domains

In [ ]:
# Evaluate on Domain A test set
print("\n" + "="*60)
print("BASELINE MODEL EVALUATION")
print("="*60)

metrics_a, pred_a, true_a, probs_a = evaluate_model(model, test_loader_a, device)

print("\nPerformance on Domain A (General News):")
for metric, value in metrics_a.items():
    print(f"  {metric.capitalize()}: {value:.4f}")

# Create test loader for Domain B
test_dataset_b = FakeNewsDataset(test_b['combined_text'].values, test_b['label'].values, tokenizer)
test_loader_b = DataLoader(test_dataset_b, batch_size=16, shuffle=False)

# Evaluate on Domain B test set
metrics_b, pred_b, true_b, probs_b = evaluate_model(model, test_loader_b, device)

print("\nPerformance on Domain B (COVID-19 News):")
for metric, value in metrics_b.items():
    print(f"  {metric.capitalize()}: {value:.4f}")

# Calculate performance drop
print("\nPerformance Drop (Domain Shift Impact):")
for metric in ['accuracy', 'f1', 'precision', 'recall']:
    drop = metrics_a[metric] - metrics_b[metric]
    print(f"  {metric.capitalize()}: {drop:.4f} ({drop*100:.2f}% decrease)")

## 8. Active Learning Strategies Implementation

In [ ]:
class ActiveLearningStrategies:
    """Collection of Active Learning selection strategies"""
    
    @staticmethod
    def random_selection(pool_indices, budget, random_state=42):
        """Random sampling strategy"""
        np.random.seed(random_state)
        return np.random.choice(pool_indices, size=min(budget, len(pool_indices)), replace=False)
    
    @staticmethod
    def least_confidence(probs, pool_indices, budget):
        """Uncertainty-based: Least Confidence sampling"""
        # Get max probability for each sample
        max_probs = np.max(probs, axis=1)
        # Select samples with lowest max probability (most uncertain)
        uncertainties = 1 - max_probs
        # Get indices of most uncertain samples
        sorted_indices = np.argsort(uncertainties)[::-1]
        selected = [pool_indices[i] for i in sorted_indices[:budget]]
        return np.array(selected)
    
    @staticmethod
    def margin_sampling(probs, pool_indices, budget):
        """Uncertainty-based: Margin sampling"""
        # Sort probabilities for each sample
        sorted_probs = np.sort(probs, axis=1)
        # Margin = difference between top two probabilities
        margins = sorted_probs[:, -1] - sorted_probs[:, -2]
        # Select samples with smallest margin (most uncertain between top 2)
        sorted_indices = np.argsort(margins)
        selected = [pool_indices[i] for i in sorted_indices[:budget]]
        return np.array(selected)
    
    @staticmethod
    def entropy_sampling(probs, pool_indices, budget):
        """Uncertainty-based: Entropy sampling"""
        # Calculate entropy for each sample
        epsilon = 1e-10
        entropy = -np.sum(probs * np.log(probs + epsilon), axis=1)
        # Select samples with highest entropy (most uncertain)
        sorted_indices = np.argsort(entropy)[::-1]
        selected = [pool_indices[i] for i in sorted_indices[:budget]]
        return np.array(selected)
    
    @staticmethod
    def diversity_clustering(embeddings, pool_indices, budget, n_clusters=None):
        """Diversity-based: K-means clustering selection"""
        if n_clusters is None:
            n_clusters = budget
        
        # Perform K-means clustering
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(embeddings)
        
        # Select one sample from each cluster (closest to centroid)
        selected = []
        for cluster_id in range(n_clusters):
            cluster_indices = np.where(cluster_labels == cluster_id)[0]
            if len(cluster_indices) > 0:
                # Select first sample from each cluster
                selected.append(pool_indices[cluster_indices[0]])
        
        return np.array(selected[:budget])
    
    @staticmethod
    def uncertainty_diversity_mixed(probs, embeddings, pool_indices, budget, alpha=0.5):
        """Mixed strategy: Combine uncertainty and diversity"""
        # Calculate uncertainty scores
        epsilon = 1e-10
        uncertainty = -np.sum(probs * np.log(probs + epsilon), axis=1)
        uncertainty_normalized = (uncertainty - uncertainty.min()) / (uncertainty.max() - uncertainty.min() + 1e-10)
        
        # Calculate diversity scores using distance to center
        center = np.mean(embeddings, axis=0)
        distances = np.linalg.norm(embeddings - center, axis=1)
        diversity_normalized = (distances - distances.min()) / (distances.max() - distances.min() + 1e-10)
        
        # Combine scores
        combined_scores = alpha * uncertainty_normalized + (1 - alpha) * diversity_normalized
        
        # Select samples with highest combined scores
        sorted_indices = np.argsort(combined_scores)[::-1]
        selected = [pool_indices[i] for i in sorted_indices[:budget]]
        return np.array(selected)
    
    @staticmethod
    def core_set_selection(embeddings, pool_indices, budget, train_embeddings=None):
        """Mixed strategy: Core-set selection using k-center greedy"""
        if train_embeddings is not None:
            # Start with train embeddings as already selected
            all_embeddings = np.vstack([train_embeddings, embeddings])
            selected_mask = np.zeros(len(all_embeddings), dtype=bool)
            selected_mask[:len(train_embeddings)] = True
        else:
            all_embeddings = embeddings
            selected_mask = np.zeros(len(all_embeddings), dtype=bool)
        
        # Greedy k-center
        for _ in range(budget):
            if selected_mask.all():
                break
            
            # Calculate minimum distance to selected set for each unselected sample
            min_distances = np.full(len(all_embeddings), np.inf)
            selected_indices = np.where(selected_mask)[0]
            
            for idx in range(len(all_embeddings)):
                if not selected_mask[idx]:
                    distances = np.linalg.norm(all_embeddings[idx] - all_embeddings[selected_indices], axis=1)
                    min_distances[idx] = distances.min()
            
            # Select the sample with maximum minimum distance
            new_idx = np.argmax(min_distances)
            selected_mask[new_idx] = True
        
        # Return indices from pool
        if train_embeddings is not None:
            pool_selected = np.where(selected_mask[len(train_embeddings):])[0]
            return pool_indices[pool_selected[:budget]]
        else:
            pool_selected = np.where(selected_mask)[0]
            return pool_indices[pool_selected[:budget]]

print("Active Learning strategies defined successfully!")

## 9. Active Learning Experiment Framework

In [ ]:
def get_embeddings_and_probs(model, dataset, device):
    """Extract embeddings and predictions from model"""
    model.eval()
    model.to(device)
    
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
    all_embeddings = []
    all_probs = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model.distilbert(input_ids=input_ids, attention_mask=attention_mask)
            # Use [CLS] token embedding
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            cls_outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(cls_outputs.logits, dim=1).cpu().numpy()
            
            all_embeddings.append(embeddings)
            all_probs.append(probs)
    
    return np.vstack(all_embeddings), np.vstack(all_probs)

def run_active_learning_experiment(
    model, tokenizer,
    train_data, pool_data, test_data,
    strategy_name, selection_fn,
    budget_ratios, n_runs=3,
    device='cuda',
    incremental=True,  # Keep previously selected samples
    keep_domain_a=True,  # Keep Domain A data
    continue_training=True  # Continue from checkpoint vs retrain
):
    """Run active learning experiment"""
    results = []
    
    for run in range(n_runs):
        print(f"\n--- Run {run + 1}/{n_runs} ---")
        
        # Initialize fresh model for each run
        model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
        model.to(device)
        
        # Initial training on Domain A
        train_dataset = FakeNewsDataset(train_data['combined_text'].values, train_data['label'].values, tokenizer)
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        optimizer = AdamW(model.parameters(), lr=2e-5)
        model = train_model(model, train_loader, optimizer, device, epochs=3)
        
        # Track selected samples
        selected_indices = np.array([], dtype=int)
        
        # Evaluate baseline (0% budget)
        test_dataset = FakeNewsDataset(test_data['combined_text'].values, test_data['label'].values, tokenizer)
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
        metrics, _, _, _ = evaluate_model(model, test_loader, device)
        results.append({
            'run': run,
            'budget_ratio': 0,
            'n_samples': 0,
            **metrics
        })
        print(f"Budget 0%: Accuracy = {metrics['accuracy']:.4f}, F1 = {metrics['f1']:.4f}")
        
        pool_indices = np.arange(len(pool_data))
        
        for budget_ratio in budget_ratios:
            # Calculate budget
            budget = int(len(pool_data) * budget_ratio)
            
            if incremental and len(selected_indices) > 0:
                # Only select additional samples to reach the budget
                remaining_budget = budget - len(selected_indices)
                if remaining_budget <= 0:
                    # Already have enough samples
                    continue
                remaining_pool = np.setdiff1d(pool_indices, selected_indices)
                
                if len(remaining_pool) == 0:
                    continue
                    
                # Get embeddings and probs for remaining pool
                remaining_data = pool_data.iloc[remaining_pool].reset_index(drop=True)
                remaining_dataset = FakeNewsDataset(
                    remaining_data['combined_text'].values,
                    remaining_data['label'].values,
                    tokenizer
                )
                embeddings, probs = get_embeddings_and_probs(model, remaining_dataset, device)
                
                # Select new samples
                if strategy_name == 'random':
                    new_selected = ActiveLearningStrategies.random_selection(
                        np.arange(len(remaining_pool)), remaining_budget, random_state=run*100+budget_ratio*1000
                    )
                else:
                    new_selected = selection_fn(probs, embeddings, np.arange(len(remaining_pool)), remaining_budget)
                
                # Map back to original indices
                new_selected = remaining_pool[new_selected]
                selected_indices = np.concatenate([selected_indices, new_selected])
            else:
                # Non-incremental: select all samples from scratch
                pool_dataset = FakeNewsDataset(
                    pool_data['combined_text'].values,
                    pool_data['label'].values,
                    tokenizer
                )
                embeddings, probs = get_embeddings_and_probs(model, pool_dataset, device)
                
                if strategy_name == 'random':
                    selected_indices = ActiveLearningStrategies.random_selection(
                        pool_indices, budget, random_state=run*100+budget_ratio*1000
                    )
                else:
                    selected_indices = selection_fn(probs, embeddings, pool_indices, budget)
            
            # Prepare new training data
            if keep_domain_a:
                new_train_data = pd.concat([
                    train_data,
                    pool_data.iloc[selected_indices]
                ], ignore_index=True)
            else:
                new_train_data = pool_data.iloc[selected_indices].copy()
            
            # Train model
            if continue_training:
                # Continue training from current checkpoint
                new_train_dataset = FakeNewsDataset(
                    new_train_data['combined_text'].values,
                    new_train_data['label'].values,
                    tokenizer
                )
                new_train_loader = DataLoader(new_train_dataset, batch_size=16, shuffle=True)
                optimizer = AdamW(model.parameters(), lr=1e-5)  # Lower learning rate for fine-tuning
                model = train_model(model, new_train_loader, optimizer, device, epochs=2)
            else:
                # Retrain from scratch
                model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
                model.to(device)
                new_train_dataset = FakeNewsDataset(
                    new_train_data['combined_text'].values,
                    new_train_data['label'].values,
                    tokenizer
                )
                new_train_loader = DataLoader(new_train_dataset, batch_size=16, shuffle=True)
                optimizer = AdamW(model.parameters(), lr=2e-5)
                model = train_model(model, new_train_loader, optimizer, device, epochs=3)
            
            # Evaluate
            metrics, _, _, _ = evaluate_model(model, test_loader, device)
            results.append({
                'run': run,
                'budget_ratio': budget_ratio,
                'n_samples': len(selected_indices),
                **metrics
            })
            print(f"Budget {budget_ratio*100:.0f}% ({len(selected_indices)} samples): Accuracy = {metrics['accuracy']:.4f}, F1 = {metrics['f1']:.4f}")
    
    return pd.DataFrame(results)

## 10. Run Active Learning Experiments

In [ ]:
# Define budget ratios
budget_ratios = [0.01, 0.02, 0.05, 0.10, 0.20, 0.50]

# Store all results
all_results = {}

# Active Learning Protocol Settings
INCREMENTAL = True  # Keep previously selected samples
KEEP_DOMAIN_A = True  # Keep Domain A data during adaptation
CONTINUE_TRAINING = True  # Continue from checkpoint

In [ ]:
# 1. Random Sampling (Baseline)
print("="*60)
print("STRATEGY 1: RANDOM SAMPLING")
print("="*60)

random_results = run_active_learning_experiment(
    model, tokenizer,
    train_a, pool_b, test_b,
    strategy_name='random',
    selection_fn=None,
    budget_ratios=budget_ratios,
    n_runs=3,
    device=device
)
all_results['random'] = random_results

In [ ]:
# 2. Uncertainty-based: Least Confidence
print("="*60)
print("STRATEGY 2: LEAST CONFIDENCE")
print("="*60)

def least_confidence_fn(probs, embeddings, pool_indices, budget):
    return ActiveLearningStrategies.least_confidence(probs, pool_indices, budget)

lc_results = run_active_learning_experiment(
    model, tokenizer,
    train_a, pool_b, test_b,
    strategy_name='least_confidence',
    selection_fn=least_confidence_fn,
    budget_ratios=budget_ratios,
    n_runs=3,
    device=device
)
all_results['least_confidence'] = lc_results

In [ ]:
# 3. Uncertainty-based: Entropy Sampling
print("="*60)
print("STRATEGY 3: ENTROPY SAMPLING")
print("="*60)

def entropy_fn(probs, embeddings, pool_indices, budget):
    return ActiveLearningStrategies.entropy_sampling(probs, pool_indices, budget)

entropy_results = run_active_learning_experiment(
    model, tokenizer,
    train_a, pool_b, test_b,
    strategy_name='entropy',
    selection_fn=entropy_fn,
    budget_ratios=budget_ratios,
    n_runs=3,
    device=device
)
all_results['entropy'] = entropy_results

In [ ]:
# 4. Diversity-based: Clustering
print("="*60)
print("STRATEGY 4: DIVERSITY CLUSTERING")
print("="*60)

def diversity_fn(probs, embeddings, pool_indices, budget):
    return ActiveLearningStrategies.diversity_clustering(embeddings, pool_indices, budget)

diversity_results = run_active_learning_experiment(
    model, tokenizer,
    train_a, pool_b, test_b,
    strategy_name='diversity',
    selection_fn=diversity_fn,
    budget_ratios=budget_ratios,
    n_runs=3,
    device=device
)
all_results['diversity'] = diversity_results

In [ ]:
# 5. Mixed Strategy: Uncertainty + Diversity
print("="*60)
print("STRATEGY 5: UNCERTAINTY + DIVERSITY (MIXED)")
print("="*60)

def mixed_fn(probs, embeddings, pool_indices, budget):
    return ActiveLearningStrategies.uncertainty_diversity_mixed(probs, embeddings, pool_indices, budget, alpha=0.5)

mixed_results = run_active_learning_experiment(
    model, tokenizer,
    train_a, pool_b, test_b,
    strategy_name='mixed',
    selection_fn=mixed_fn,
    budget_ratios=budget_ratios,
    n_runs=3,
    device=device
)
all_results['mixed'] = mixed_results

In [ ]:
# 6. Mixed Strategy: Core-Set Selection
print("="*60)
print("STRATEGY 6: CORE-SET SELECTION")
print("="*60)

def core_set_fn(probs, embeddings, pool_indices, budget):
    return ActiveLearningStrategies.core_set_selection(embeddings, pool_indices, budget)

core_set_results = run_active_learning_experiment(
    model, tokenizer,
    train_a, pool_b, test_b,
    strategy_name='core_set',
    selection_fn=core_set_fn,
    budget_ratios=budget_ratios,
    n_runs=3,
    device=device
)
all_results['core_set'] = core_set_results

## 11. Results Aggregation and Analysis

In [ ]:
# Aggregate results across runs
def aggregate_results(results_df):
    """Calculate mean and std for each budget ratio"""
    return results_df.groupby('budget_ratio').agg({
        'accuracy': ['mean', 'std'],
        'f1': ['mean', 'std'],
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std']
    }).reset_index()

aggregated = {}
for strategy, df in all_results.items():
    aggregated[strategy] = aggregate_results(df)

# Display summary table
print("\n" + "="*80)
print("RESULTS SUMMARY: ACCURACY BY STRATEGY AND BUDGET")
print("="*80)

summary_data = []
for strategy, df in aggregated.items():
    for _, row in df.iterrows():
        summary_data.append({
            'Strategy': strategy,
            'Budget': f"{row['budget_ratio']*100:.0f}%",
            'Accuracy': f"{row[('accuracy', 'mean')]:.4f} ± {row[('accuracy', 'std')]:.4f}",
            'F1': f"{row[('f1', 'mean')]:.4f} ± {row[('f1', 'std')]:.4f}"
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

## 12. Visualization: Performance vs Cost Curves

In [ ]:
# Create performance vs cost curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color scheme for strategies
colors = {
    'random': '#1f77b4',
    'least_confidence': '#ff7f0e',
    'entropy': '#2ca02c',
    'diversity': '#d62728',
    'mixed': '#9467bd',
    'core_set': '#8c564b'
}

labels = {
    'random': 'Random',
    'least_confidence': 'Least Confidence',
    'entropy': 'Entropy Sampling',
    'diversity': 'Diversity Clustering',
    'mixed': 'Uncertainty + Diversity',
    'core_set': 'Core-Set'
}

# Plot Accuracy vs Budget
for strategy, df in aggregated.items():
    axes[0].errorbar(
        df['budget_ratio'] * 100,
        df[('accuracy', 'mean')],
        yerr=df[('accuracy', 'std')],
        label=labels[strategy],
        color=colors[strategy],
        marker='o',
        capsize=3,
        linewidth=2
    )

axes[0].set_xlabel('Budget (%)', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs Annotation Budget', fontsize=14)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# Plot F1 vs Budget
for strategy, df in aggregated.items():
    axes[1].errorbar(
        df['budget_ratio'] * 100,
        df[('f1', 'mean')],
        yerr=df[('f1', 'std')],
        label=labels[strategy],
        color=colors[strategy],
        marker='o',
        capsize=3,
        linewidth=2
    )

axes[1].set_xlabel('Budget (%)', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('F1 Score vs Annotation Budget', fontsize=14)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('/my-project/download/performance_vs_cost.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved to: /my-project/download/performance_vs_cost.png")

In [ ]:
# Create detailed comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(budget_ratios))
width = 0.12

for i, (strategy, df) in enumerate(aggregated.items()):
    acc_values = [df[df['budget_ratio'] == br][('accuracy', 'mean')].values[0] for br in budget_ratios]
    ax.bar(x + i * width, acc_values, width, label=labels[strategy], color=colors[strategy])

ax.set_xlabel('Budget Ratio', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy Comparison Across Strategies and Budgets', fontsize=14)
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels([f'{br*100:.0f}%' for br in budget_ratios])
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/my-project/download/strategy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved to: /my-project/download/strategy_comparison.png")

## 13. Analysis of Selected Samples

In [ ]:
# Analyze characteristics of selected samples
def analyze_selected_samples(pool_data, selected_indices, strategy_name):
    """Analyze characteristics of selected samples"""
    selected_data = pool_data.iloc[selected_indices]
    
    print(f"\n{'='*60}")
    print(f"ANALYSIS: {strategy_name.upper()}")
    print(f"{'='*60}")
    
    print(f"\nLabel distribution:")
    print(f"  Fake News: {sum(selected_data['label'] == 1)} ({sum(selected_data['label'] == 1)/len(selected_data)*100:.1f}%)")
    print(f"  Real News: {sum(selected_data['label'] == 0)} ({sum(selected_data['label'] == 0)/len(selected_data)*100:.1f}%)")
    
    # Text length analysis
    selected_data['text_length'] = selected_data['combined_text'].apply(len)
    print(f"\nText length statistics:")
    print(f"  Mean: {selected_data['text_length'].mean():.0f} characters")
    print(f"  Std: {selected_data['text_length'].std():.0f} characters")
    
    return selected_data

# Sample analysis for 10% budget
print("Sample Analysis for Budget = 10%:")
for strategy in ['random', 'entropy', 'mixed']:
    df = all_results[strategy]
    # Get indices from first run with 10% budget
    run_data = df[(df['run'] == 0) & (df['budget_ratio'] == 0.10)]
    print(f"\nStrategy: {strategy}")
    print(f"Results available for analysis")

## 14. Statistical Significance Analysis

In [ ]:
from scipy import stats

# Compare each strategy with random baseline
print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TEST (t-test vs Random)")
print("="*80)

for budget_ratio in budget_ratios:
    print(f"\nBudget: {budget_ratio*100:.0f}%")
    print("-" * 40)
    
    random_accs = all_results['random'][all_results['random']['budget_ratio'] == budget_ratio]['accuracy'].values
    
    for strategy in ['least_confidence', 'entropy', 'diversity', 'mixed', 'core_set']:
        strategy_accs = all_results[strategy][all_results[strategy]['budget_ratio'] == budget_ratio]['accuracy'].values
        
        if len(random_accs) > 1 and len(strategy_accs) > 1:
            t_stat, p_value = stats.ttest_ind(random_accs, strategy_accs)
            significance = "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
            print(f"  {strategy}: t={t_stat:.3f}, p={p_value:.4f} {significance}")

## 15. Save Results

In [ ]:
# Save all results to CSV
results_dir = '/my-project/download/'

for strategy, df in all_results.items():
    df.to_csv(f"{results_dir}{strategy}_results.csv", index=False)

# Save aggregated results
with pd.ExcelWriter(f"{results_dir}all_results.xlsx") as writer:
    for strategy, df in aggregated.items():
        df.to_excel(writer, sheet_name=strategy, index=False)

print(f"Results saved to {results_dir}")
print("\nFiles generated:")
for f in os.listdir(results_dir):
    if f.endswith('.csv') or f.endswith('.xlsx') or f.endswith('.png'):
        print(f"  - {f}")

## 16. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

print("\n1. DOMAIN SHIFT IMPACT:")
print(f"   Baseline accuracy on Domain A: {metrics_a['accuracy']:.4f}")
print(f"   Baseline accuracy on Domain B: {metrics_b['accuracy']:.4f}")
print(f"   Performance drop: {(metrics_a['accuracy'] - metrics_b['accuracy'])*100:.2f}%")

print("\n2. BEST PERFORMING STRATEGY:")
# Find best strategy at 10% budget
best_acc = 0
best_strategy = None
for strategy, df in aggregated.items():
    acc_10 = df[df['budget_ratio'] == 0.10][('accuracy', 'mean')].values[0]
    if acc_10 > best_acc:
        best_acc = acc_10
        best_strategy = strategy

print(f"   At 10% budget: {best_strategy} with accuracy {best_acc:.4f}")

print("\n3. ACTIVE LEARNING PROTOCOL:")
print(f"   - Incremental selection: {INCREMENTAL}")
print(f"   - Keep Domain A data: {KEEP_DOMAIN_A}")
print(f"   - Continue training: {CONTINUE_TRAINING}")

print("\n4. KEY FINDINGS:")
print("   - Active learning strategies show varying effectiveness")
print("   - Domain adaptation improves with more labeled target data")
print("   - Uncertainty-based methods tend to select informative samples")